In [1]:
import numpy as np
from matplotlib import pyplot as plt
from cycler import cycler
plt.rc('axes', prop_cycle=cycler(color=['0.0', '0.4', '0.8']))
import matplotlib; matplotlib.rcParams.update({'font.size': 18})

In [2]:
from pywrangling.jupyter_slides_functions import apply_beamer_theme
from pywrangling.graphing_functions import use_plotplainblind, set_top_ylabel
from IPython.display import HTML, display
import course_helpers as chh
apply_beamer_theme(
    "madrid",
    skip_font_download=True,
    author="Robert Pettis",
    date="Fall 2026",
    show_slide_number=True,
    cell_tags=True,
    notebook_path="14b-Checkpoint-Quiz.ipynb",
)

<IPython.core.display.Javascript object>

No cells with recognised display tags found.


In [3]:
# Presenter clicker remap. Guarded so a student's Restart & Run All
# never breaks on it if pywrangling is missing.
try:
    from pywrangling.jupyter_slides_functions import enable_reveal_key_remap
    enable_reveal_key_remap({
        38: "left",         # Up arrow   -> previous  (clicker left button)
        40: "right",        # Down arrow -> next      (clicker right button)
        27: "up",           # Escape     -> up        (clicker play button)
        66: "down",         # B          -> down      (clicker stop button)
        116: None,          # F5         -> block refresh
        9:  "togglePause",  # Tab        -> black screen
    })
except Exception:
    pass  # not presenting, or pywrangling not installed

<div class="beamer-meta" data-author="Robert Pettis" data-date="Fall 2026"></div>

# Checkpoint


<br>

<b style="color:#73000A;">Robert Pettis</b>
Fall 2026

## Before We Start: Updating Your Repo

<span style="display:inline-block; background:#73000A; color:#fff; padding:2px 12px; border-radius:8px; font-weight:700; font-size:.9em;">Reminder</span> &nbsp;same git rhythm as the setup lecture.

| command | what it does |
|---|---|
| `git pull upstream main` | grab today's lecture before class |
| `git checkout -b ps1` | a branch for your own work |
| `git add .` then `git commit -m "..."` | snapshot your changes |
| `git push origin ps1` | send it to your fork, then open a pull request |
| `git status` | what changed, any time |

### Question 1: Nine Days and Five Days

The blue company has 9 days of prices and the red company has 5, and we want one
network that predicts day 10 for both.

What does the feedback loop buy us?

- <b style="color:#73000A;">A.</b> The longer series gets more weights, so the network has more to work with
- <b style="color:#73000A;">B.</b> Unroll one copy per day, 9 copies or 5, and every copy shares the same weights and biases
- <b style="color:#73000A;">C.</b> The red company's series is padded out to 9 days with zeros
- <b style="color:#73000A;">D.</b> One network is trained per company, and the feedback loop is how they are combined

### Question 2: Two Days That End the Same Way

Low then medium predicts about $1$. High then medium predicts $0$. Today's input
is $0.5$ in both, and the weights are the same $1.8$, $-0.5$ and $1.1$.

Where does the difference come from?

- <b style="color:#73000A;">A.</b> $w_1$ is larger in the second copy of the network than in the first
- <b style="color:#73000A;">B.</b> Yesterday's ReLU output goes around the loop through $w_2 = -0.5$. High gives $1.8 \times -0.5 = -0.9$, which cancels today's $0.5 \times 1.8 = 0.9$
- <b style="color:#73000A;">C.</b> ReLU sets the second day to zero whenever yesterday was high
- <b style="color:#73000A;">D.</b> The bias on the output differs between the two cases

### Question 3: From Two Days to Three

We had two copies, for yesterday and today. Now we want three days, so we unroll
twice and plug in $1$, then $0.5$, then $0.5$.

What happens to the number of weights and biases we have to train?

- <b style="color:#73000A;">A.</b> It grows with the number of days, which is why long series are expensive
- <b style="color:#73000A;">B.</b> It stays the same. All three copies share the same $w_1$, $b_1$, $w_2$ and output weight
- <b style="color:#73000A;">C.</b> One new bias is added per copy, so three days means two extra biases
- <b style="color:#73000A;">D.</b> It doubles each time we unroll, because each copy feeds the next

### Question 4: Four Days, One Derivative

Take the two weight RNN with no activation and no biases. With 4 days we unroll
3 times, and the oldest value is $D_{-3}$.

In $d\,\text{Predicted} / d\,W_1$, what multiplies $D_{-3}$?

- <b style="color:#73000A;">A.</b> $3 \times W_2$
- <b style="color:#73000A;">B.</b> $W_2^3$, because the value passes through $W_2$ once per unroll
- <b style="color:#73000A;">C.</b> $W_2^4$, one factor per day of data
- <b style="color:#73000A;">D.</b> $W_1^3$, since the derivative is taken with respect to $W_1$

### Question 5: Keep $w_2$ Below 1?

With 21 days and $w_2 = 2$, the oldest value is multiplied by $2^{20} = 1{,}048{,}576$.
A student suggests fixing that by always keeping $w_2$ between $-1$ and $1$.

What happens?

- <b style="color:#73000A;">A.</b> That fixes it. Any $w_2$ inside $[-1, 1]$ trains fine however far we unroll
- <b style="color:#73000A;">B.</b> It trades one problem for the other. At $w_2 = 0.5$ the oldest value is multiplied by $0.5^{20} = 0.000001$ and stops influencing training
- <b style="color:#73000A;">C.</b> Nothing changes, because the gradient depends on $W_1$, not $w_2$
- <b style="color:#73000A;">D.</b> The gradient vanishes only when $w_2$ is negative, so keeping it positive and below 1 is safe

### Question 6: What the LSTM Does Differently

An LSTM runs two paths through the unit instead of one.

Why does that help with the gradient?

- <b style="color:#73000A;">A.</b> The LSTM has more weights, so the gradients it produces are larger
- <b style="color:#73000A;">B.</b> The Long-Term Memory crosses the unit with no weight or bias on it. It is scaled by a percentage and added to, so it does not pick up a factor of $w_2$ per unroll
- <b style="color:#73000A;">C.</b> The LSTM caps every gradient at $1$ before the update
- <b style="color:#73000A;">D.</b> The LSTM does not unroll, so there is nothing to multiply repeatedly

### Question 7: The Forget Gate on a Big Negative

Previous Long-Term Memory $2$, Previous Short-Term Memory $1$, and now an input
of $-10$. Stage 1 computes

$$(1 \times 2.70) + (-10 \times 1.63) + 1.62 = -11.98, \qquad f(-11.98) = 0.00$$

What is the Long-Term Memory after stage 1?

- <b style="color:#73000A;">A.</b> $2 - 11.98 = -9.98$
- <b style="color:#73000A;">B.</b> $0$. The sigmoid gives about $0$, and $2 \times 0 = 0$, so the long-term memory is forgotten
- <b style="color:#73000A;">C.</b> $2$, unchanged, because a sigmoid output cannot be negative
- <b style="color:#73000A;">D.</b> $-11.98$, which becomes the new long-term memory

### Question 8: Sigmoid in Three Places, Tanh in Two

Inside the unit, the three percentage blocks use the sigmoid and the two
potential memories use tanh.

Why those two, in those places?

- <b style="color:#73000A;">A.</b> The sigmoid runs from $0$ to $1$, which is what a percentage to keep needs. Tanh runs from $-1$ to $1$, so a potential memory can push the long-term memory either way
- <b style="color:#73000A;">B.</b> They are interchangeable. Either one would do in any of the five places
- <b style="color:#73000A;">C.</b> Tanh trains faster, so it is used wherever the derivative matters most
- <b style="color:#73000A;">D.</b> The sigmoid is only there because the final output has to be a probability

### Question 9: Two Companies, One Difference

Company A and Company B have the same values on days 2, 3 and 4. We run days 1
to 4 through the same unrolled LSTM, same weights, and the outputs are $0.0$ for
A and $1.0$ for B, which are the right day 5 values.

Which is right?

- <b style="color:#73000A;">A.</b> Each company got its own trained LSTM, which is why the answers differ
- <b style="color:#73000A;">B.</b> Day 1 was carried in the Long-Term Memory, the cell state, which ends at $0.0$ for A and $2.4$ for B. The output is the final Short-Term Memory, the hidden state
- <b style="color:#73000A;">C.</b> The Short-Term Memory carried it, since that is the path the output is read from
- <b style="color:#73000A;">D.</b> The weights changed on day 5 to account for the difference